In [1]:
import numpy as np
from pathlib import Path
np.random.seed(96)

## The Core Interfaces `AnalogCore` and `CrossSimParameters`

The core simulation object in CrossSim is the `AnalogCore`. All of the examples you see in this tutorial internally wrap the basic `AnalogCore` interface. `AnalogCore` is specifically designed to be drop-in compatible with NumPy (or CuPy for GPU-accelerated simulation) arrays.

To demonstrate this interface we'll start by using the matrix-vector multiplications (MVM).

In [2]:
# Define a random matrix W
N1, N2 = 50, 100
W = np.random.uniform(-1,1,size=(N2,N1))

# Define a random vector x
x = np.random.uniform(-1,1,size=(N1,))

# Compute the MVM in NumPy:
y_np = W @ x

`AnalogCore`s are made up of a 1D or 2D NumPy array and a set of parameters using the `CrossSimParameters` object.

`CrossSimParameters` is a structured tree of objects containing all of the parameters for a simulation. We'll go over many of the functionalities throughout this tutorial but the basic structure is below.
```
CrossSimParameters
    ->core         # Properties of the data mapping strategies (e.g., negative numbers)
    ->simulation   # Properties about how the simulation will run (e.g., useGPU)
    ->xbar         # Properties of the simulated analog arrays
        ->adc
        ->array
        ->dac
        ->device
```

By convention the default value for each property is *ideal* behavior so if you don't set something you can assume it behaves ideally.

In [3]:
# Import Simulator Interfaces
from simulator import AnalogCore, CrossSimParameters

# Create a default (ideal) parameters object and an AnalogCore from that object
params = CrossSimParameters()
xbar = AnalogCore(W, params=params)

# Use the AnalogCore like a NumPy Array
y_xsim = xbar @ x

Once created, the AnalogCore can be treated as if it were the familiar NumPy array, except that when multiplying by the AnalogCore, an analog MVM is being simulated!

Now let's compare CrossSim's MVM result with NumPy and plot the result. Since we assumed ideal analog hardware, they should match exactly.

In [4]:
assert(np.allclose(y_np,y_xsim))
print("--- MVM check passed! ---")

--- MVM check passed! ---


## Other `AnalogCore` Functionality
`AnalogCore` is designed to cover the full range of NumPy multiplication operations including left and right multiplies, transpose operations, matrix-matrix multiplies, and operations using subsets of a matrix (i.e., NumPy slices).

It is up to the user to ensure that such operations are physically appropriate to the simulated hardware. For matrix multiplies CrossSim assumes that the operation will be implemented as a series of MVM operations and handles decomposing the operation into MVMs internally. Similarly for submatrix operations CrossSim internally handles padding the input with zeros and/or masking the output.

In [5]:
## Multiply by the transpose of the array
x = np.random.uniform(-1,1,size=(N2,))
y_t_np = W.T @ x
y_t_xsim = xbar.T @ x

assert(np.allclose(y_t_np,y_t_xsim))
print("--- Transpose MVM check passed! ---")

--- Transpose MVM check passed! ---


In [6]:
## Multiplying from the opposite direction (VMM)
y_t_np = x @ W
y_t_xsim = x @ xbar

assert(np.allclose(y_t_np,y_t_xsim))
print("--- VMM check passed! ---")

--- VMM check passed! ---


In [7]:
## Multiplying with Subarrays
x = np.random.uniform(-1,1,20)
y_np = W[:20, :20] @ x
y_xsim = xbar[:20, :20] @ x

assert(np.allclose(y_np,y_xsim))
print("--- Subarray check passed! ---")

--- Subarray check passed! ---


In [8]:
## Matrix Multiplication
# Generate a matrix to be programmed into the array
W = np.random.uniform(-1,1,size=(N2,N1))

# Generate a matrix of input values with the appropriate dimensions
X = np.random.uniform(-1,1,size=(N1,N1))

# Compute the MatMul in NumPy
Y_np = W @ X

# Create and program an AnalogCore
xbar = AnalogCore(W, params=params)

# Compute the MatMul in CrossSim
Y_xsim = xbar @ X

# We relax the check slightly for this test
assert(np.allclose(Y_np,Y_xsim,atol=1e-6))
print("--- Left MatMul check passed! ---")

--- Left MatMul check passed! ---


In [9]:
# Generate a matrix of input values with the appropriate dimensions
X = np.random.uniform(-1,1,size=(N2,N2))

# Compute the MatMul in NumPy
Y_np_2 = X @ W

# Compute the MatMul in CrossSim
Y_xsim_2 = X @ xbar

# We relax the check slightly for this test
assert(np.allclose(Y_np_2,Y_xsim_2,atol=1e-6))
print("--- Right MatMul check passed! ---")

--- Right MatMul check passed! ---


There are other important properties of `AnalogCore`s including updating (reprogramming) the stored either in full or in part, and changing the native datatype, but this basic functionality will be sufficient to demonstrate a wide range of CrossSim simulation features.

## Interacting with `CrossSimParameters`
`CrossSimParameters` behave as nested objects and can be updated using simple `foo.bar` syntax.

In older tutorials and resources you may see some specialized setter functions for configuring parameters objections. These are now deprecated and may be removed in future versions. For the remainder of this tutorial we will use the `.` syntax below.

**Important Note:** When updating `CrossSimParameters` objects `AnalogCore` objects should be recreated. Updating the internal `AnalogCore.params` object is *unsupported* and may result in unintended behavior.

In [10]:
# Build a new params object to modify
params_noisy = CrossSimParameters()

# Turn on a device noise model with a normally distributed 1% error
params_noisy.xbar.device.programming_error.enable = True
params_noisy.xbar.device.programming_error.model = "NormalIndependentDevice"
params_noisy.xbar.device.programming_error.magnitude = 0.01

# Build a new AnalogCore from the parameters object
xbar_noisy = AnalogCore(W, params=params_noisy)

# Compute a *noisy* Matmul using CrossSim
Y_xsim_noisy = X @ xbar_noisy

print(Y_np_2[:5,:5]- Y_xsim_noisy[:5,:5])

[[ 0.10242622 -0.0807051  -0.06699278  0.09817713 -0.0009628 ]
 [-0.0126073  -0.0472936   0.11211075  0.03331327 -0.00075557]
 [ 0.17768751 -0.02905134 -0.00333126  0.01689101  0.06518333]
 [-0.11830078  0.09208706 -0.07982191 -0.08623482 -0.10116679]
 [ 0.18905735 -0.00205888  0.04036066  0.02719164  0.04531087]]


Because of the programming error, our analog result no longer matches the NumPy result. This is how we use CrossSim to analyze the impact of analog non-idealities.

With sufficiently complex `CrossSimParameters` objects it can be useful to save and load known good configurations. This allows simulation portability and replicability. Lets just save out our existing config to JSON and load it back in.

In [11]:
# Save our noisy params file to a file
params_noisy.to_json(Path("./params/params_noisy.json"))

# Create a new parameters object
params_new = CrossSimParameters()

# Load the parameters from the JSON file
params_new = params_new.from_json("./params/params_noisy.json")

With these basics, you are fully equipped to start doing complex experiments in CrossSim!